<div align="center">
  <img src="assets/Day13.png" alt="Databricks 14 Days AI Challenge - Day 13" width="800"/>
</div>

## DAY 13 (21/01/26) – Model Comparison & Pipelines

### 📚 Learning Objectives
A single model is rarely the best answer. To build robust AI, we must test multiple approaches and standardize the workflow. Today we cover:
* **Model Tournament:** Training multiple algorithms (Linear, Tree, Random Forest) on the same data to pick a winner.
* **Spark ML Pipelines:** Moving away from "script-based" ML to "pipeline-based" ML (Assembler -> Model). 
* **Scalability:** Why we switch from Scikit-Learn (Single Node) to Spark ML (Distributed) when data grows.

### 🚀 Strategy: "The Bake-Off"
1.  **The Tournament (Sklearn):** We will loop through 3 distinct algorithms using our Pandas dataframe. We use **MLflow** to log all results automatically.
2.  **The Analysis:** We identify the "Champion" model based on R2 Score.
3.  **The Production Build (Spark ML):** We re-implement a robust pipeline using native Spark ML libraries. This ensures that even if our data grows to 100TB, our training won't crash.

###Data Setup & Helper Functions
**Task**: Load data and prepare for the tournament. **Concept**: We reuse the Gold data. For the Sklearn part, we convert to Pandas. For Spark ML, we keep it as a Spark DataFrame.

In [0]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Setup Context
spark.sql("USE CATALOG course_catalog")
spark.sql("USE SCHEMA ecommerce_governed")

# 1. Load Data
# We use the Gold table (Product Performance)
df_gold_spark = spark.table("gold_product_perf")
df_gold_pandas = df_gold_spark.toPandas().fillna(0)

# 2. Features & Target
# Goal: Predict 'unique_purchases' based on 'unique_views'
# Note: In a real scenario, we would add more features like 'avg_price', 'brand_popularity', etc.
X = df_gold_pandas[["unique_views"]]
y = df_gold_pandas["unique_purchases"]

# 3. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Data Ready. Training on {len(X_train)} products.")

###The Model Tournament (Sklearn)
**Task**: Train 3 models and compare metrics. **Concept**: Instead of writing code 3 times, we iterate through a dictionary of models. MLflow organizes them into a clean comparison view.

In [0]:
# ---------------------------------------------------------
# STEP 1: DEFINE CONTENDERS
# ---------------------------------------------------------
models = {
    "Linear_Regression": LinearRegression(),
    "Decision_Tree": DecisionTreeRegressor(max_depth=5),
    "Random_Forest": RandomForestRegressor(n_estimators=100, max_depth=5)
}

mlflow.set_experiment("/Users/engineeringltctanmay@gmail.com/Day13_Model_Tournament")

print("🏆 STARTING MODEL TOURNAMENT...")

# ---------------------------------------------------------
# STEP 2: RUN TOURNAMENT
# ---------------------------------------------------------
for name, model in models.items():
    with mlflow.start_run(run_name=f"Model_{name}"):
        # A. Log Hyperparameters
        mlflow.log_param("model_type", name)
        
        # B. Train
        model.fit(X_train, y_train)
        
        # C. Predict & Evaluate
        score = model.score(X_test, y_test)
        
        # D. Log Metrics & Model
        mlflow.log_metric("r2_score", score)
        mlflow.sklearn.log_model(model, "model")
        
        print(f"   🥊 {name:20} | R2 Score: {score:.4f}")

print("\n🏁 Tournament Complete. Check MLflow UI for charts.")

<div align="center">
  <img src="assets/Day13_Winner.png" alt="Comparing 3 ML models - Day 13">
</div>

###Spark ML Pipeline (The Production Build)
**Task**: Build a Spark ML Pipeline. **Concept**: Scikit-Learn works great on small data (RAM constrained). **Spark ML** works on massive data (Distributed).

* **VectorAssembler**: Spark ML requires all input features to be squeezed into a single "Vector" column.

* **Pipeline**: Chains the transformer (Assembler) and estimator (Model) into one savable object.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression as SparkLR
from pyspark.ml.evaluation import RegressionEvaluator

# ---------------------------------------------------------
# STEP 3: BUILD SCALABLE PIPELINE
# ---------------------------------------------------------

# 1. Define Stages
# Stage A: Transform features into a Vector
assembler = VectorAssembler(
    inputCols=["unique_views"], 
    outputCol="features"
)

# Stage B: Define the Model
lr_spark = SparkLR(
    featuresCol="features", 
    labelCol="unique_purchases",
    maxIter=10,
    regParam=0.3
)

# 2. Build Pipeline
pipeline = Pipeline(stages=[assembler, lr_spark])

# 3. Split Data (Native Spark Split)
train_df, test_df = df_gold_spark.fillna(0).randomSplit([0.8, 0.2], seed=42)

# 4. Fit Pipeline
print("⚙️ Training Spark ML Pipeline...")
pipeline_model = pipeline.fit(train_df)

# 5. Make Predictions
predictions = pipeline_model.transform(test_df)

# ---------------------------------------------------------
# EVALUATION & VISUALIZATION
# ---------------------------------------------------------
evaluator = RegressionEvaluator(
    labelCol="unique_purchases", 
    predictionCol="prediction", 
    metricName="r2"
)
r2_spark = evaluator.evaluate(predictions)

print(f"✅ Spark ML Model R2 Score: {r2_spark:.4f}")

print("📊 Prediction vs Actuals (Top 5):")
display(predictions.select("product_id", "unique_views", "unique_purchases", "prediction").limit(10))

### 🥇 Selecting the Best Model

**Strategy:**
Now that we have trained both local (Sklearn) and distributed (Spark) models, we look at the results.

1.  **Check MLflow:** Look for the run with the highest `r2_score` (likely Random Forest due to non-linear relationships).
2.  **Check Latency:** Linear Regression is faster to train but might be less accurate.
3.  **Check Scale:** If our dataset grows to 1 Billion rows, we **must** choose the Spark ML pipeline, even if its R2 is slightly lower than a complex Sklearn Random Forest, because the Sklearn model will run out of RAM (OOM Error).

**Verdict for this dataset:**
* **Accuracy Winner:** Random Forest (Captures complex view-to-purchase patterns).
* **Scale Winner:** Spark ML Linear Regression (Can scale to petabytes).

### 🧠 Key Learnings & Takeaways
* **Automation:** We used a loop to train multiple models instantly. This is the foundation of AutoML.
* **Spark Vectors:** Unlike Pandas, Spark ML requires a `VectorAssembler` step. This compresses features into a dense vector format optimized for network transfer across the cluster.
* **Pipelines:** A Pipeline (`Assembler` -> `Model`) is safer than separate steps because it guarantees the *exact same transformations* are applied to Training data and Test data (preventing skew).